https://www.kaggle.com/code/karan842/customer-segmentation-rfm-eda-knn-95

https://fiqey.medium.com/customer-segmentation-using-knn-with-rfm-analysis-52c7fd65b8d1

# The goal is to retain a dataset of valid customers.
So we opt to drop those records where no customer ID is present.

In [1]:
import pandas as pd, os, numpy as np, matplotlib.pyplot as plt,re

cur_dir = os.path.abspath('')
dataset1_path = os.path.join(cur_dir,'..','Datasets/online+retail/online_retail_I.xlsx')
dataset2_path = os.path.join(cur_dir,'..','Datasets/online+retail/online_retail_II.xlsx')

In [2]:
df1 = pd.read_excel(dataset1_path)
df2 = pd.read_excel(dataset2_path)

In [3]:
print(f"{'Columns of each dataset':<100}{'# of columns':<12}\n","-"*100)
print(f"{','.join(df1.columns.to_list()):<100}|",f"{df1.columns.__len__():<12}")
print("-"*100)
print(f"{','.join(df2.columns.to_list()):<100}|",f"{df2.columns.__len__():<12}")

Columns of each dataset                                                                             # of columns
 ----------------------------------------------------------------------------------------------------
Unnamed: 0,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country        | 9           
----------------------------------------------------------------------------------------------------
Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country                        | 8           


##### Making the two data have consistent columns

In [4]:
if df1.columns.__len__()>df2.columns.__len__():
    df1.drop(df1.columns[0],axis=1,inplace=True)

In [5]:
df2.rename({columns[0]:columns[1] for columns in zip(df2.columns,df1.columns)},axis=1,inplace=True)

In [6]:
print(f"{'Columns of each dataset':<100}{'# of columns':<12}\n","-"*100)
print(f"{','.join(df1.columns.to_list()):<100}|",f"{df1.columns.__len__():<12}")
print("-"*100)
print(f"{','.join(df2.columns.to_list()):<100}|",f"{df2.columns.__len__():<12}")

Columns of each dataset                                                                             # of columns
 ----------------------------------------------------------------------------------------------------
InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country                   | 8           
----------------------------------------------------------------------------------------------------
InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country                   | 8           


In [7]:
joined_df = pd.concat((df1,df2), ignore_index=True)

In [8]:
joined_df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067365,538171,22271,FELTCRAFT DOLL ROSIE,2,2010-12-09 20:01:00,2.95,17530.0,United Kingdom
1067366,538171,22750,FELTCRAFT PRINCESS LOLA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
1067367,538171,22751,FELTCRAFT PRINCESS OLIVIA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
1067368,538171,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-09 20:01:00,3.75,17530.0,United Kingdom


#### Making all the strings in "StockCode" and "Description" upper case

In [9]:
joined_df["Description"] = joined_df["Description"].apply(lambda x: x.upper() if isinstance(x,str) else str(x) if isinstance(x,int) else x)
joined_df["StockCode"] = joined_df["StockCode"].apply(lambda x: x.upper() if isinstance(x,str) else str(x) if isinstance(x,int) else x)

# Processing the NaN values

##### Which columns have NaN values

In [10]:
print(f"{'column':<30}|{"# of NaN"}\n"+"-"*100)

a = joined_df.isna().sum()

for i,j in zip(a.index.to_list(),a.values):
    print(f"{i:<30}|{j}")

column                        |# of NaN
----------------------------------------------------------------------------------------------------
InvoiceNo                     |0
StockCode                     |0
Description                   |4382
Quantity                      |0
InvoiceDate                   |0
UnitPrice                     |0
CustomerID                    |243007
Country                       |0


##### We first deal with the Description column using the StockCodes
- In case any other sales with this stock exists and has a valid Desc. -> replace the NaN

In [11]:
multi_desc_codes = joined_df.groupby("StockCode")["Description"].nunique(dropna=True).loc[lambda x: x>1].index

stock_desc_map = joined_df[ joined_df["StockCode"].isin(multi_desc_codes) ]\
           .dropna(subset=["Description"])\
           .groupby("StockCode")["Description"]\
           .first()

mask_stock = joined_df["StockCode"].isin(multi_desc_codes)

joined_df.loc[mask_stock,"Description"] = joined_df.loc[mask_stock,"StockCode"].map(stock_desc_map)

##### We check for columns with NaN values again

In [12]:
print(f"{'column':<30}|{"# of NaN"}\n"+"-"*100)

a = joined_df.isna().sum()

for i,j in zip(a.index.to_list(),a.values):
    print(f"{i:<30}|{j}")

column                        |# of NaN
----------------------------------------------------------------------------------------------------
InvoiceNo                     |0
StockCode                     |0
Description                   |3425
Quantity                      |0
InvoiceDate                   |0
UnitPrice                     |0
CustomerID                    |243007
Country                       |0


##### We drop the remaining NaN values of Description

In [13]:
joined_df.dropna(subset="Description",inplace=True)

##### Next we replace the NaN of CustomerID with -1, indicating one-time purchases

In [14]:
joined_df.loc[joined_df.CustomerID.isna(),"CustomerID"] = -1

# Changing the name ERIE to Ireland

In [15]:
joined_df.loc[joined_df["Country"]=="EIRE","Country"]="Ireland"

# Dealing with Invalid Descriptions 
Our preprocessing should be done at this point. But maybe we can identify invalid values of Description which would make our task easier in the Analysis step.

In [16]:
print('List of Descriptions with one word')
print("-"*100)
oneword = sorted(joined_df.loc[
    (joined_df["Description"].str.split(" |,").str.len() == 1)
,"Description"].unique())
print(oneword)
print("-"*100)

print('List of Descriptions with two words')
print("-"*100)
twowords = sorted(joined_df.loc[
    (joined_df["Description"].str.split(" |,").str.len() == 2)
,"Description"].unique())
print(twowords)



List of Descriptions with one word
----------------------------------------------------------------------------------------------------
['?', 'ADJUSTMENT', 'CARRIAGE', 'CHECK', 'DAMAGED', 'DAMAGES', 'DISCOUNT', 'EBAY', 'FOUND', 'LOST', 'MAILOUT', 'MANUAL', 'POSTAGE', 'SAMPLES']
----------------------------------------------------------------------------------------------------
List of Descriptions with two words
----------------------------------------------------------------------------------------------------
['AMAZON FEE', 'ANIMAL STICKERS', 'BANK CHARGES', 'BATHROOM HOOK', 'BINGO SET', 'BUTTERFLIES STICKERS', 'CAPIZ CHANDELIER', 'CARNIVAL BRACELET', 'CHAMBRE HOOK', 'CHILLI LIGHTS', 'CHOCOLATE CALCULATOR', 'CHRYSANTHEMUM NOTEBOOK', 'CONCH SHELL', 'CONGRATULATIONS BUNTING', 'CRUK COMMISSION', 'DAISY RING', 'DIAMANTE NECKLACE', 'DOGGY RUBBER', 'DOLPHIN WINDMILL', 'DOORMAT HEARTS', 'DOORMAT TOPIARY', 'DOTCOM POSTAGE', 'EMPIRE TEDDY', 'FLAMINGO LIGHTS', 'FLOWER WINDMILL', 'FROG CANDLE',

In [17]:
keywords = oneword+['AMAZON FEE','BANK CHARGES','WET DAMAGED', 'THROW AWAY', 'THROWN AWAY', 'THROWN AWAY-RUSTY','PACKING CHARGE','GIVEN AWAY','DOTCOM POSTAGE',]

joined_df.loc[joined_df["Description"].isin(keywords)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
45,536370,POST,POSTAGE,3,2010-12-01 08:45:00,18.00,12583.0,France
141,C536379,D,DISCOUNT,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
386,536403,POST,POSTAGE,1,2010-12-01 11:27:00,15.00,12791.0,Netherlands
1123,536527,POST,POSTAGE,1,2010-12-01 13:04:00,18.00,12662.0,Germany
1423,536540,C2,CARRIAGE,1,2010-12-01 14:05:00,50.00,14911.0,Ireland
...,...,...,...,...,...,...,...,...
1066796,538148,DOT,DOTCOM POSTAGE,1,2010-12-09 16:26:00,547.32,-1.0,United Kingdom
1066900,538149,84968E,CHECK,1,2010-12-09 16:27:00,25.49,-1.0,United Kingdom
1066909,538149,DOT,DOTCOM POSTAGE,1,2010-12-09 16:27:00,620.68,-1.0,United Kingdom
1067035,538153,DOT,DOTCOM POSTAGE,1,2010-12-09 16:31:00,822.94,-1.0,United Kingdom


In [18]:
customer_special_keywords = joined_df.loc[ (joined_df["Description"].isin(keywords))
    &
    (joined_df["CustomerID"]!=-1),
    "Description"].unique().tolist()

to_remove_keywords = [word for word in oneword+twowords if word not in customer_special_keywords]

In [19]:
joined_df = joined_df.loc[~(joined_df.Description.isin(to_remove_keywords))]

In [22]:
joined_df = joined_df.sort_values(by="InvoiceDate",ignore_index=True)

In [24]:
joined_df.to_pickle(os.path.join(cur_dir,'..','Datasets/online+retail/online_retail_joined.pkl'))

import pickle as pkl

with open(os.path.join(cur_dir,'..','Datasets/online+retail/special_keywords.pkl'),'wb') as f:
    pkl.dump(customer_special_keywords,f)